# GDELT V2Tone: Sentiment Score Calculation and Final Feature Concatenation

The aim of this Notebook is to extract sentiment features (V2Tone) from the 3GB `mapped_100k_12m.parquet` news mapping table,
Using the CSI (Comprehensive Sentiment Index) algorithm, a score is calculated for each individual article; these scores are aggregated by company and ultimately seamlessly integrated into the Theme feature table already completed by team members.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import time

BASE_FEATURE_CSV = "gkg_features_theme_score_100k.csv"
MAPPED_NEWS_PARQUET = "mapped_100k_12m.parquet"

FINAL_COMPLETE_CSV = "final_company_features_100k.csv"
FINAL_COMPLETE_PQ = "final_company_features_100k.parquet"

print("Path configuration complete")

Path configuration complete


## CSI Algorithm
Trim V2Tone directly at row level and smooth it to the range [-1, 1] using the `Tanh` function. If the word count is less than 100 words, apply a weight reduction penalty.

In [3]:
def calculate_article_csi(tone_str):
    """Calculate the Comprehensive Sentiment Index (CSI) for a single article based on the V2Tone string"""
    if pd.isna(tone_str) or not str(tone_str).strip():
        return np.nan

    parts = str(tone_str).split(',')
    if len(parts) < 7:
        return np.nan

    try:
        tone = float(parts[0])
        pos = float(parts[1])
        neg = float(parts[2])
        word_count = float(parts[6])

        # Normalisation of the base tone
        base_tone = tone / 10.0

        # Net Sentiment Ratio for Noise Reduction
        net_sentiment = (pos - neg) / (pos + neg + 0.0001)

        # Preliminary weighted score
        csi_raw = (base_tone * 0.6) + (net_sentiment * 0.4)

        # Length-based credibility denoising multiplier
        if word_count < 100:
            multiplier = 0.5  # Reduced weighting of short messages
        elif word_count > 500:
            multiplier = 1.2  # In-depth reporting weighting
        else:
            multiplier = 1.0

        # Tanh 挤压函数，将分数严格限制在 [-1, 1] 之间
        return np.tanh(csi_raw * multiplier)
    except Exception:
        return np.nan

print("CSI success")

CSI success


## Memory-optimised reading and single-article score calculation
When dealing with a 3GB Parquet file, we only read the two columns we need (`company_search_name` and `V2Tone`). This prevents memory crashes.

In [4]:
print("Reading the news mapping table (extracting key columns only)...")
start_time = time.time()

df_news = pd.read_parquet(
    MAPPED_NEWS_PARQUET,
    columns=['company_search_name', 'V2Tone']
)

# Cleaning up empty data
df_news = df_news.dropna(subset=['V2Tone', 'company_search_name'])
print(f"Loaded successfully {len(df_news)} Valid news records Time taken {time.time() - start_time:.2f} seconds")

print("Calculating the CSI sentiment score for a single article...")
df_news['CSI_Article_Score'] = df_news['V2Tone'].apply(calculate_article_csi)
df_news.head()

Reading the news mapping table (extracting key columns only)...
Loaded successfully 4599644 Valid news records Time taken 1.20 seconds
Calculating the CSI sentiment score for a single article...


,company_search_name,V2Tone,CSI_Article_Score
0,human services,"-7.92079207920792,2.14521452145215,10.06600660...",-0.707243
1,universal pictures,"3.7037037037037,3.7037037037037,0,3.7037037037...",0.552666
2,human services,"-3.35497835497836,3.03030303030303,6.385281385...",-0.390671
3,pravo,"5.69105691056911,7.31707317073171,1.6260162601...",0.534201
4,creative content agency,"3.28947368421053,4.27631578947368,0.9868421052...",0.419730


## By company, aggregated average sentiment score
Calculate the overall macro-sentiment for each company, and once the calculation is complete, manually delete the 3GB table to free up memory.

In [5]:
print("Aggregating average sentiment scores by company level...")
company_tone_agg = df_news.groupby('company_search_name').agg(
    avg_csi_score=('CSI_Article_Score', 'mean')
).reset_index()

print(f"A total of {len(company_tone_agg)} Companies with distinct emotional characteristics")

del df_news

company_tone_agg.head()

Aggregating average sentiment scores by company level...
A total of 5710 Companies with distinct emotional characteristics


,company_search_name,avg_csi_score
0,100318,0.305021
1,1201,-0.019757
2,1336,-0.010660
3,14930,0.008052
4,1933,-0.016246


## Merging and Saving the Final Feature Table
Append the aggregated sentiment scores to the main Theme feature table using a `Left Join`. For companies with no news, enter 0.0.

In [8]:
print("Loading the Theme benchmark table and performing a Left Join...")
df_base = pd.read_csv(BASE_FEATURE_CSV)
print(f"The benchmark table contains {len(df_base)} company")

df_final = df_base.merge(
    company_tone_agg,
    left_on='search_name',
    right_on='company_search_name',
    how='left'
).drop(columns=['company_search_name'])

# Imputing missing values: The sentiment score for companies with no news is recorded as 0.0
df_final['has_sentiment_score'] = df_final['avg_csi_score'].notna()
df_final['avg_csi_score'] = df_final['avg_csi_score'].fillna(0.0)

cols = df_final.columns.tolist()

cols.remove('has_sentiment_score')
cols.remove('avg_csi_score')
if 'theme_score' in cols:
    cols.remove('theme_score')

cols.extend(['has_sentiment_score', 'theme_score', 'avg_csi_score'])
df_final = df_final[cols]


print("Saving the final, complete feature width table...")
df_final.to_csv(FINAL_COMPLETE_CSV, index=False)
df_final.to_parquet(FINAL_COMPLETE_PQ, compression='snappy')

print(f"CSV save in: {FINAL_COMPLETE_CSV}")
print("\n--- Preview of the final output table data ---")
df_final.head()

Loading the Theme benchmark table and performing a Left Join...
The benchmark table contains 100000 company
Saving the final, complete feature width table...
CSV save in: final_company_features_100k.csv

--- Preview of the final output table data ---


,CompanyNumber,CompanyName,search_name,primary_sector,Accounts_AccountCategory,business_article_count,has_business_news,gkg_features_available,likely_ambiguous,has_sentiment_score,theme_score,avg_csi_score
0,13209628,TICKETY BOO TEETH LIMITED,tickety boo teeth,Healthcare,UNAUDITED ABRIDGED,0,False,False,False,False,NaN,0.0
1,13887674,TECH INFUSION LIMITED,tech infusion,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False,False,False,NaN,0.0
2,13408899,THE SCC ACADEMY LIMITED,the scc academy,Fast growth & emerging sector,TOTAL EXEMPTION FULL,0,False,False,False,False,NaN,0.0
3,SC374368,GRACEFRUIT LIMITED,gracefruit,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False,False,False,NaN,0.0
4,13675979,LINHAM LIMITED,linham,Wholesale & Retail,TOTAL EXEMPTION FULL,0,False,False,True,False,NaN,0.0
